# DistilBERT Humanitarian Category Classifier — Full Training

**Project:** Disaster Information Intelligence and Decision Support System  
**Task:** 10-class humanitarian category classification on QCRI/HumAID-all  
**Model:** `distilbert-base-uncased` (fine-tuned from pretrained weights)  
**Primary metric:** Macro F1 (severe class imbalance — 56:1 ratio)  

## Dataset splits (verified)
| Split | Rows |
|---|---:|
| train | 53,531 |
| validation | 7,793 |
| test | 15,160 |
| **Total** | **76,484** |

## Classes (10)
1. caution_and_advice
2. displaced_people_and_evacuations
3. infrastructure_and_utility_damage
4. injured_or_dead_people
5. missing_or_found_people
6. not_humanitarian
7. other_relevant_information
8. requests_or_urgent_needs
9. rescue_volunteering_or_donation_effort
10. sympathy_and_support

## Important
- Validation is used **only** for early stopping and checkpoint selection.
- The test split is touched **once** at the end for final evaluation.
- Validation metrics ≠ test metrics — they are reported separately.
- All outputs go to `/kaggle/working/distilbert_humanitarian/`


---
## Cell 1 — GPU Check


In [ ]:
import subprocess, sys

# GPU availability
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('nvidia-smi:', result.stdout.strip() if result.returncode == 0 else 'not available')

import torch
print(f'PyTorch      : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be very slow on CPU.')
    print('Enable GPU in Kaggle: Settings → Accelerator → GPU T4 x2 or P100')

print(f'Python       : {sys.version.split()[0]}')

---
## Cell 2 — Install Dependencies


In [ ]:
# Kaggle already has: torch, numpy, pandas, matplotlib, seaborn, scikit-learn
# We only need to ensure datasets and transformers are up-to-date.
# Pin versions for reproducibility — these are compatible with Kaggle's base image.

import subprocess

pkgs = [
    'datasets==2.20.0',
    'transformers==4.44.2',
    'accelerate==0.34.2',
]

for pkg in pkgs:
    print(f'Installing {pkg} ...')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q', '--no-deps'],
        capture_output=True, text=True
    )
    # fall back to with-deps if no-deps fails
    if r.returncode != 0:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                       check=True)
    print(f'  OK')

print('\nAll packages ready.')

---
## Cell 3 — Imports


In [ ]:
import re
import sys
import json
import time
import shutil
import warnings
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset

from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
print('All imports OK')

---
## Cell 4 — Configuration

All tunable parameters are defined here. Edit this cell only.


In [ ]:
# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR   = Path('/kaggle/working/distilbert_humanitarian')
CKPT_DIR     = OUTPUT_DIR / 'best_model'      # HuggingFace model + tokenizer
RESULTS_DIR  = OUTPUT_DIR / 'results'          # JSON metrics
FIGURES_DIR  = OUTPUT_DIR / 'figures'          # PNG plots
for d in (CKPT_DIR, RESULTS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Pretrained model ──────────────────────────────────────────────────────────
PRETRAINED = 'distilbert-base-uncased'

# ── Training hyperparameters ──────────────────────────────────────────────────
# Tuned for Kaggle T4 (16 GB) / P100 (16 GB) / V100 (16 GB).
# For T4 x2 you can increase BATCH_SIZE to 64.
EPOCHS       = 5          # max epochs (early stopping may fire earlier)
BATCH_SIZE   = 32         # per-device batch size
GRAD_ACCUM   = 2          # effective batch = 32 × 2 = 64
LR           = 2e-5       # AdamW peak learning rate
WEIGHT_DECAY = 0.01       # AdamW weight decay
MAX_LEN      = 128        # token sequence length (covers >95pct of HumAID tweets)
WARMUP_RATIO = 0.06       # fraction of total steps for linear warmup
PATIENCE     = 2          # early stopping: epochs without val Macro F1 improvement
RANDOM_SEED  = 42

# ── Reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Label definitions (canonical order — must match humaid_loader.py) ────────
HUMAID_CLASSES = [
    'caution_and_advice',
    'displaced_people_and_evacuations',
    'infrastructure_and_utility_damage',
    'injured_or_dead_people',
    'missing_or_found_people',
    'not_humanitarian',
    'other_relevant_information',
    'requests_or_urgent_needs',
    'rescue_volunteering_or_donation_effort',
    'sympathy_and_support',
]
LABEL2ID = {lbl: i for i, lbl in enumerate(HUMAID_CLASSES)}
ID2LABEL  = {i: lbl for i, lbl in enumerate(HUMAID_CLASSES)}
N_LABELS  = len(HUMAID_CLASSES)

print(f'Device       : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
print(f'Output dir   : {OUTPUT_DIR}')
print(f'Pretrained   : {PRETRAINED}')
print(f'Epochs (max) : {EPOCHS}')
print(f'Batch size   : {BATCH_SIZE}  grad_accum={GRAD_ACCUM}  effective={BATCH_SIZE*GRAD_ACCUM}')
print(f'LR           : {LR}')
print(f'Max length   : {MAX_LEN}')
print(f'Patience     : {PATIENCE}')
print(f'Num classes  : {N_LABELS}')

---
## Cell 5 — Text Cleaning Pipeline

Exact copy of `preprocessing/text_cleaner.py::clean_text()`.  
Do not simplify — must match the local project for reproducibility.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Inline copy of preprocessing/text_cleaner.py
# Source of truth: preprocessing/text_cleaner.py in the project repo.
# This copy must be kept in sync with that file.
# ─────────────────────────────────────────────────────────────────────────────

import re, unicodedata

_RE_URL          = re.compile(r'https?://\S+|ftp://\S+|www\.\S+', re.IGNORECASE)
_RE_MENTION      = re.compile(r'@\w+:?')
_RE_HASHTAG      = re.compile(r'#(\w+)')
_RE_HTML_ENTITY  = re.compile(r'&(?:[a-z]+|#\d+|#x[\da-f]+);', re.IGNORECASE)
_RE_WHITESPACE   = re.compile(r'\s+')
_RE_REPEAT_PUNCT = re.compile(r'([!?.,;:\-])\1{2,}')
_RE_EMOJI        = re.compile(
    '['
    '\U0001F600-\U0001F64F'
    '\U0001F300-\U0001F5FF'
    '\U0001F680-\U0001F6FF'
    '\U0001F1E0-\U0001F1FF'
    '\U00002702-\U000027B0'
    '\U000024C2-\U0001F251'
    '\U0001F900-\U0001F9FF'
    '\U00002500-\U00002BEF'
    '\U00010000-\U0010FFFF'
    ']+',
    flags=re.UNICODE,
)
_RE_RT_PREFIX    = re.compile(r'^RT\s+', re.IGNORECASE)

_HTML_ENTITY_MAP = {
    'amp': '&', 'lt': '<', 'gt': '>', 'quot': '"',
    'apos': "'", 'nbsp': ' ', 'ndash': '\u2013', 'mdash': '\u2014',
}

def _decode_html_entity(match):
    raw  = match.group(0)
    name = raw[1:-1]
    if name.startswith('#x') or name.startswith('#X'):
        try: return chr(int(name[2:], 16))
        except ValueError: return raw
    elif name.startswith('#'):
        try: return chr(int(name[1:]))
        except ValueError: return raw
    return _HTML_ENTITY_MAP.get(name.lower(), raw)

def clean_text(text, *, remove_emojis=True, remove_rt_prefix=True,
               decode_html_entities=True):
    """Clean a single tweet. Exact logic mirrors preprocessing/text_cleaner.py."""
    # 0. Null / non-string safety
    if text is None: return ''
    if not isinstance(text, str):
        try: text = str(text)
        except Exception: return ''
    text = text.strip()
    if not text: return ''
    # 1. HTML entity decoding
    if decode_html_entities:
        text = _RE_HTML_ENTITY.sub(_decode_html_entity, text)
    # 2. Unicode NFKC normalisation
    text = unicodedata.normalize('NFKC', text)
    # 3. RT prefix removal
    if remove_rt_prefix:
        text = _RE_RT_PREFIX.sub('', text)
    # 4. URL removal
    text = _RE_URL.sub('', text)
    # 5. @mention removal (+ trailing colon from 'RT @handle:')
    text = _RE_MENTION.sub('', text)
    # 6. Hashtag word preservation (#FloodAlert → floodalert)
    text = _RE_HASHTAG.sub(r'\1', text)
    # 7. Emoji removal
    if remove_emojis:
        text = _RE_EMOJI.sub('', text)
    # 8. Lowercase
    text = text.lower()
    # 9. Repeated punctuation (3+ → 2)
    text = _RE_REPEAT_PUNCT.sub(r'\1\1', text)
    # 10. Whitespace normalisation
    text = _RE_WHITESPACE.sub(' ', text).strip()
    return text

# Sanity-check a handful of cases
_CHECKS = [
    ('RT @WHO: #FloodAlert https://t.co/abc 235 dead',
     'floodalert 235 dead'),
    ('Aid workers &amp; volunteers needed',
     'aid workers & volunteers needed'),
    ('SOS!!!!! help NOW???',
     'sos!! help now??'),
    (None, ''),
    ('', ''),
]
for raw, expected in _CHECKS:
    got = clean_text(raw)
    status = 'OK' if got == expected else f'FAIL (got {repr(got)!r})'
    print(f'  [{status}]  {repr(raw)[:50]}')

print('\nText cleaner ready.')

---
## Cell 6 — Load and Verify HumAID Dataset


In [ ]:
print('Loading QCRI/HumAID-all from Hugging Face ...')
print('(First run downloads ~21 MB; subsequent runs use local cache.)')

CACHE_DIR = Path('/kaggle/working/humaid_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

ds = load_dataset(
    'QCRI/HumAID-all',
    cache_dir=str(CACHE_DIR),
    verification_mode='no_checks',
)

# Decode ClassLabel integers → string names
features = ds['train'].features
raw_splits = {}
for split in ('train', 'validation', 'test'):
    df = ds[split].to_pandas()
    if hasattr(features['class_label'], 'names'):
        names = features['class_label'].names
        df['class_label'] = df['class_label'].apply(
            lambda x: names[x] if isinstance(x, int) else x
        )
    raw_splits[split] = df[['tweet_text', 'class_label']].copy()

print('\n── Split verification ──')
EXPECTED = {'train': 53531, 'validation': 7793, 'test': 15160}
all_ok = True
for split, df in raw_splits.items():
    row_ok   = len(df) == EXPECTED[split]
    col_ok   = list(df.columns) == ['tweet_text', 'class_label']
    null_ok  = df['tweet_text'].isnull().sum() == 0
    labels   = sorted(df['class_label'].unique())
    lbl_ok   = labels == HUMAID_CLASSES
    ok       = row_ok and col_ok and null_ok and lbl_ok
    all_ok   = all_ok and ok
    print(f'  {split:12s}: {len(df):,} rows  '
          f'rows={"OK" if row_ok else "FAIL"}  '
          f'cols={"OK" if col_ok else "FAIL"}  '
          f'nulls={"OK" if null_ok else "FAIL"}  '
          f'labels={"OK" if lbl_ok else "FAIL"}')

assert all_ok, 'Dataset verification failed — check output above.'

print('\n── Class distribution (train) ──')
dist = raw_splits['train']['class_label'].value_counts().sort_values(ascending=False)
for lbl, cnt in dist.items():
    bar = '█' * int(cnt / 300)
    print(f'  {lbl:<50} {cnt:>6,}  ({100*cnt/len(raw_splits["train"]):5.1f}%)  {bar}')

print(f'\nImbalance ratio: {dist.max()/dist.min():.1f}:1')
print('\nDataset loaded and verified.')

---
## Cell 7 — Preprocess and Compute Class Weights


In [ ]:
print('Applying text cleaning ...')
t0 = time.time()

splits = {}
for split, df in raw_splits.items():
    d = df.copy()
    d['clean']    = d['tweet_text'].apply(clean_text)
    d['label_id'] = d['class_label'].map(LABEL2ID)
    # Validate — catch any unmapped labels immediately
    n_null = d['label_id'].isnull().sum()
    if n_null > 0:
        unknown = d.loc[d['label_id'].isnull(), 'class_label'].unique()
        raise ValueError(f'[{split}] {n_null} unmapped labels: {unknown}')
    # Validate — no empty cleaned texts
    n_empty = (d['clean'].str.strip() == '').sum()
    splits[split] = d
    print(f'  {split:12s}: {len(d):,} rows  empty_after_clean={n_empty}')

elapsed = time.time() - t0
print(f'  Cleaning took {elapsed:.1f}s')

# ── Class weights for imbalance handling ──────────────────────────────────────
# Uses sklearn compute_class_weight('balanced') — same as local training script.
# The weights are passed to CrossEntropyLoss during training and validation.
train_label_ids = splits['train']['label_id'].tolist()
class_weights_np = compute_class_weight(
    'balanced',
    classes=np.arange(N_LABELS),
    y=train_label_ids,
)
CLASS_WEIGHTS = torch.tensor(class_weights_np, dtype=torch.float).to(DEVICE)

print('\n── Class weights ──')
for i, (lbl, w) in enumerate(zip(HUMAID_CLASSES, class_weights_np)):
    print(f'  {lbl:<50} {w:.4f}')
print(f'\n  Min weight : {class_weights_np.min():.4f}')
print(f'  Max weight : {class_weights_np.max():.4f}')
print(f'  Ratio      : {class_weights_np.max()/class_weights_np.min():.1f}x')
print('\nPreprocessing complete.')

---
## Cell 8 — Tokenise and Build DataLoaders


In [ ]:
class TweetDataset(Dataset):
    """Tokenised tweet dataset for DistilBERT."""
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_len,
            return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx],
        }

print(f'Tokenising with {PRETRAINED} (max_len={MAX_LEN}) ...')
TOKENIZER = DistilBertTokenizerFast.from_pretrained(PRETRAINED)

def make_loader(split_name, shuffle):
    df = splits[split_name]
    ds_obj = TweetDataset(
        df['clean'].tolist(),
        df['label_id'].tolist(),
        TOKENIZER, MAX_LEN,
    )
    return DataLoader(
        ds_obj,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=(DEVICE.type == 'cuda'),
    )

train_loader = make_loader('train',      shuffle=True)
val_loader   = make_loader('validation', shuffle=False)
test_loader  = make_loader('test',       shuffle=False)

print(f'  Train batches      : {len(train_loader)}')
print(f'  Validation batches : {len(val_loader)}')
print(f'  Test batches       : {len(test_loader)}')
print('Tokenisation complete.')

---
## Cell 9 — Model, Optimiser, Scheduler


In [ ]:
print(f'Loading {PRETRAINED} from HuggingFace ...')
print('(Starting from pretrained weights — NOT the local smoke-test checkpoint.)')

MODEL = DistilBertForSequenceClassification.from_pretrained(
    PRETRAINED,
    num_labels=N_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
MODEL.to(DEVICE)

n_params    = sum(p.numel() for p in MODEL.parameters())
n_trainable = sum(p.numel() for p in MODEL.parameters() if p.requires_grad)
print(f'  Parameters  : {n_params/1e6:.1f}M total  {n_trainable/1e6:.1f}M trainable')

# ── Optimiser ─────────────────────────────────────────────────────────────────
OPTIMIZER = AdamW(MODEL.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# ── LR scheduler: linear warmup then linear decay ────────────────────────────
TOTAL_STEPS  = (len(train_loader) // GRAD_ACCUM) * EPOCHS
WARMUP_STEPS = int(TOTAL_STEPS * WARMUP_RATIO)
SCHEDULER    = get_linear_schedule_with_warmup(
    OPTIMIZER,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_STEPS,
)

# ── Loss function (class-weighted) ────────────────────────────────────────────
LOSS_FN = torch.nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

print(f'  Optimiser steps : {TOTAL_STEPS}  warmup={WARMUP_STEPS}')
print('Model ready.')

---
## Cell 10 — Evaluation Helper


In [ ]:
def evaluate(model, loader, device, loss_fn):
    """
    Evaluate model on a DataLoader.
    Returns a dict with loss, accuracy, macro_f1, weighted_f1,
    macro_precision, macro_recall, plus raw preds and labels for
    downstream use.
    """
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_b       = batch['labels'].to(device)
            outputs        = model(input_ids=input_ids,
                                   attention_mask=attention_mask)
            loss           = loss_fn(outputs.logits, labels_b)
            total_loss    += loss.item()
            preds          = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels_b.cpu().numpy())

    return {
        'loss':             total_loss / len(loader),
        'accuracy':         float(accuracy_score(all_labels, all_preds)),
        'macro_f1':         float(f1_score(all_labels, all_preds,
                                           average='macro',    zero_division=0)),
        'weighted_f1':      float(f1_score(all_labels, all_preds,
                                           average='weighted', zero_division=0)),
        'macro_precision':  float(precision_score(all_labels, all_preds,
                                                  average='macro', zero_division=0)),
        'macro_recall':     float(recall_score(all_labels, all_preds,
                                               average='macro', zero_division=0)),
        'preds':  all_preds,
        'labels': all_labels,
    }

print('Evaluation helper defined.')

---
## Cell 11 — Training Loop

- Validates after every epoch on the **validation split** (not test).
- Saves checkpoint only when validation Macro F1 improves.
- Stops early after `PATIENCE` consecutive epochs without improvement.
- Prints elapsed time per epoch.


In [ ]:
print('=' * 65)
print('Starting full training run')
print(f'  Max epochs : {EPOCHS}  patience={PATIENCE}')
print(f'  Train size : {len(splits["train"]):,}  steps/epoch={len(train_loader)}')
print('=' * 65)

history    = []       # per-epoch metrics (validation only; test is NOT touched here)
best_f1    = 0.0
best_epoch = 0
no_improve = 0
t_run_start = time.time()

for epoch in range(1, EPOCHS + 1):
    MODEL.train()
    epoch_loss = 0.0
    OPTIMIZER.zero_grad()
    t_epoch = time.time()

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels_b       = batch['labels'].to(DEVICE)

        outputs = MODEL(input_ids=input_ids, attention_mask=attention_mask)
        loss    = LOSS_FN(outputs.logits, labels_b) / GRAD_ACCUM
        loss.backward()
        epoch_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(MODEL.parameters(), max_norm=1.0)
            OPTIMIZER.step()
            SCHEDULER.step()
            OPTIMIZER.zero_grad()

    avg_train_loss = epoch_loss / len(train_loader)

    # ── Validation (uses val split only — test is NOT touched) ────────────────
    val_metrics = evaluate(MODEL, val_loader, DEVICE, LOSS_FN)

    epoch_time = (time.time() - t_epoch) / 60
    total_time = (time.time() - t_run_start) / 60

    row = {
        'epoch':               epoch,
        'train_loss':          round(avg_train_loss,                  4),
        'val_loss':            round(val_metrics['loss'],             4),
        'val_accuracy':        round(val_metrics['accuracy'],         4),
        'val_macro_f1':        round(val_metrics['macro_f1'],         4),
        'val_weighted_f1':     round(val_metrics['weighted_f1'],      4),
        'val_macro_precision': round(val_metrics['macro_precision'],  4),
        'val_macro_recall':    round(val_metrics['macro_recall'],     4),
        'epoch_time_min':      round(epoch_time, 2),
        'total_elapsed_min':   round(total_time, 2),
    }
    history.append(row)

    print(f'Epoch {epoch}/{EPOCHS}  '
          f'train_loss={row["train_loss"]:.4f}  '
          f'val_loss={row["val_loss"]:.4f}  '
          f'val_macro_f1={row["val_macro_f1"]:.4f}  '
          f'val_acc={row["val_accuracy"]:.4f}  '
          f'[{epoch_time:.1f} min / {total_time:.1f} min total]')

    # ── Checkpoint on improvement ─────────────────────────────────────────────
    if val_metrics['macro_f1'] > best_f1:
        best_f1    = val_metrics['macro_f1']
        best_epoch = epoch
        no_improve = 0
        MODEL.save_pretrained(CKPT_DIR)
        TOKENIZER.save_pretrained(CKPT_DIR)
        print(f'  ✓ New best  val Macro F1={best_f1:.4f}  checkpoint saved → {CKPT_DIR}')
    else:
        no_improve += 1
        print(f'  · No improvement ({no_improve}/{PATIENCE})')
        if no_improve >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch} '
                  f'(no val Macro F1 improvement for {PATIENCE} epochs).')
            break

TOTAL_TRAIN_TIME = (time.time() - t_run_start) / 60
print(f'\nTraining complete.')
print(f'  Best epoch         : {best_epoch}')
print(f'  Best val Macro F1  : {best_f1:.4f}  (VALIDATION — not test)')
print(f'  Total training time: {TOTAL_TRAIN_TIME:.1f} min')
print(f'  Checkpoint saved to: {CKPT_DIR}')

---
## Cell 12 — Final Test Evaluation

**The test split is evaluated exactly once, here.**  
The best checkpoint (selected on validation Macro F1) is loaded fresh.  
Validation metrics from training are NOT used as test metrics.


In [ ]:
print('Loading best checkpoint for final test evaluation ...')
print(f'  Checkpoint : {CKPT_DIR}')
print(f'  Best epoch : {best_epoch}  (val Macro F1={best_f1:.4f})')
print('  NOTE: test split has NOT been seen during training.')

best_model = DistilBertForSequenceClassification.from_pretrained(CKPT_DIR)
best_model.to(DEVICE)

test_metrics = evaluate(best_model, test_loader, DEVICE, LOSS_FN)

print('\n' + '=' * 65)
print('FINAL TEST SET RESULTS  (held-out, evaluated once)')
print('=' * 65)
print(f'  Accuracy         : {test_metrics["accuracy"]:.4f}')
print(f'  Macro F1         : {test_metrics["macro_f1"]:.4f}  ← primary metric')
print(f'  Weighted F1      : {test_metrics["weighted_f1"]:.4f}')
print(f'  Macro Precision  : {test_metrics["macro_precision"]:.4f}')
print(f'  Macro Recall     : {test_metrics["macro_recall"]:.4f}')
print('=' * 65)

# ── Per-class F1 ──────────────────────────────────────────────────────────────
per_class_f1_arr = f1_score(
    test_metrics['labels'], test_metrics['preds'],
    average=None, labels=list(range(N_LABELS)), zero_division=0,
)
PER_CLASS_F1 = {HUMAID_CLASSES[i]: round(float(per_class_f1_arr[i]), 4)
                for i in range(N_LABELS)}

print('\n── Per-class F1 (test) ──')
for lbl, f1 in sorted(PER_CLASS_F1.items(), key=lambda x: -x[1]):
    bar    = '█' * int(f1 * 20)
    marker = '  ← minority' if lbl == 'missing_or_found_people' else ''
    print(f'  {lbl:<50} {f1:.4f}  {bar}{marker}')

# ── Full classification report ────────────────────────────────────────────────
CLS_REPORT = classification_report(
    test_metrics['labels'], test_metrics['preds'],
    target_names=HUMAID_CLASSES, zero_division=0,
)
print('\n── Full classification report (test) ──')
print(CLS_REPORT)

# ── Comparison: best val vs test ──────────────────────────────────────────────
print('── Validation vs Test (sanity check — these are DIFFERENT splits) ──')
best_val_row = next(r for r in history if r['epoch'] == best_epoch)
print(f'  Best val Macro F1  : {best_val_row["val_macro_f1"]:.4f}')
print(f'  Test  Macro F1     : {test_metrics["macro_f1"]:.4f}')
print(f'  Best val Accuracy  : {best_val_row["val_accuracy"]:.4f}')
print(f'  Test  Accuracy     : {test_metrics["accuracy"]:.4f}')

---
## Cell 13 — Save All Results


In [ ]:
print('Saving results ...')

# ── training_config.json ──────────────────────────────────────────────────────
training_config = {
    'pretrained':      PRETRAINED,
    'max_len':         MAX_LEN,
    'batch_size':      BATCH_SIZE,
    'grad_accum':      GRAD_ACCUM,
    'effective_batch': BATCH_SIZE * GRAD_ACCUM,
    'lr':              LR,
    'weight_decay':    WEIGHT_DECAY,
    'warmup_ratio':    WARMUP_RATIO,
    'epochs_max':      EPOCHS,
    'epochs_run':      history[-1]['epoch'],
    'best_epoch':      best_epoch,
    'patience':        PATIENCE,
    'seed':            RANDOM_SEED,
    'smoke_test':      False,
    'class_imbalance': 'compute_class_weight(balanced) via CrossEntropyLoss weight',
    'device':          str(DEVICE),
    'gpu_name':        (torch.cuda.get_device_name(0)
                        if DEVICE.type == 'cuda' else 'CPU'),
    'train_size':      len(splits['train']),
    'val_size':        len(splits['validation']),
    'test_size':       len(splits['test']),
    'total_train_time_min': round(TOTAL_TRAIN_TIME, 2),
}
(OUTPUT_DIR / 'training_config.json').write_text(
    json.dumps(training_config, indent=2), encoding='utf-8'
)
print(f'  Saved: {OUTPUT_DIR / "training_config.json"}')

# ── transformer_results.json ─────────────────────────────────────────────────
results_summary = {
    'model':                  PRETRAINED,
    'task':                   'humanitarian_category',
    'smoke_test':             False,
    'note':                   'FULL TRAINING RUN — Kaggle GPU',
    'training_config':        training_config,
    'best_val_macro_f1':      round(best_f1, 4),
    'best_epoch':             best_epoch,
    'validation_history':     history,
    'test': {
        'accuracy':           round(test_metrics['accuracy'],        4),
        'macro_f1':           round(test_metrics['macro_f1'],        4),
        'weighted_f1':        round(test_metrics['weighted_f1'],     4),
        'macro_precision':    round(test_metrics['macro_precision'], 4),
        'macro_recall':       round(test_metrics['macro_recall'],    4),
    },
    'per_class_f1_test':      PER_CLASS_F1,
    'classification_report_test': CLS_REPORT,
}
(RESULTS_DIR / 'transformer_results.json').write_text(
    json.dumps(results_summary, indent=2), encoding='utf-8'
)
print(f'  Saved: {RESULTS_DIR / "transformer_results.json"}')

# ── classification_report.txt ─────────────────────────────────────────────────
(RESULTS_DIR / 'classification_report_test.txt').write_text(
    f'DistilBERT Humanitarian Classifier — Test Classification Report\n'
    f'Best epoch: {best_epoch}  Best val Macro F1: {best_f1:.4f}\n'
    f'(These are TEST set results — validation is reported separately)\n\n'
    + CLS_REPORT,
    encoding='utf-8',
)
print(f'  Saved: {RESULTS_DIR / "classification_report_test.txt"}')

print('\nAll results saved.')

---
## Cell 14 — Training Curves


In [ ]:
epochs_done = [r['epoch']         for r in history]
train_losses= [r['train_loss']    for r in history]
val_losses  = [r['val_loss']      for r in history]
val_mf1     = [r['val_macro_f1']  for r in history]
val_wf1     = [r['val_weighted_f1'] for r in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_done, train_losses, marker='o', label='Train loss')
axes[0].plot(epochs_done, val_losses,   marker='s', label='Val loss')
axes[0].axvline(best_epoch, color='gray', linestyle='--',
                linewidth=0.9, label=f'best epoch={best_epoch}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()

axes[1].plot(epochs_done, val_mf1, marker='o', label='Val Macro F1')
axes[1].plot(epochs_done, val_wf1, marker='s', label='Val Weighted F1')
axes[1].axvline(best_epoch, color='gray', linestyle='--', linewidth=0.9)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1')
axes[1].set_title('Validation F1 Scores  (val only — not test)')
axes[1].legend()

plt.suptitle(
    f'DistilBERT Humanitarian Classifier — Training History\n'
    f'Best val Macro F1={best_f1:.4f} at epoch {best_epoch}  '
    f'Test Macro F1={test_metrics["macro_f1"]:.4f}',
    fontsize=12, fontweight='bold',
)
plt.tight_layout()
path = FIGURES_DIR / 'training_curves.png'
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

---
## Cell 15 — Confusion Matrix


In [ ]:
cm = confusion_matrix(
    test_metrics['labels'], test_metrics['preds'],
    labels=list(range(N_LABELS)),
)
short_labels = [l.replace('_', '\n') for l in HUMAID_CLASSES]

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short_labels, yticklabels=short_labels,
            ax=ax, annot_kws={'size': 8})
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title(
    f'DistilBERT Humanitarian Classifier — Test Confusion Matrix\n'
    f'Test Macro F1={test_metrics["macro_f1"]:.4f}  '
    f'Accuracy={test_metrics["accuracy"]:.4f}',
    fontsize=12,
)
plt.xticks(fontsize=7, rotation=45, ha='right')
plt.yticks(fontsize=7, rotation=0)
plt.tight_layout()
path = FIGURES_DIR / 'confusion_matrix_test.png'
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

---
## Cell 16 — Per-Class F1 Chart


In [ ]:
f1_values = [PER_CLASS_F1[c] for c in HUMAID_CLASSES]
colors = ['#d62728' if f < 0.5 else '#ff7f0e' if f < 0.7 else '#2ca02c'
          for f in f1_values]
short = [l.replace('_', '\n') for l in HUMAID_CLASSES]

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(range(N_LABELS), f1_values, color=colors)
ax.set_xticks(range(N_LABELS))
ax.set_xticklabels(short, fontsize=7, rotation=45, ha='right')
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='F1=0.5')
ax.axhline(0.7, color='blue', linestyle=':', linewidth=0.8, label='F1=0.7')
ax.set_title('DistilBERT — Per-Class F1 (Test Split)  '
             f'Macro F1={test_metrics["macro_f1"]:.4f}')
ax.legend(fontsize=9)
for i, f in enumerate(f1_values):
    ax.text(i, f + 0.02, f'{f:.2f}', ha='center', fontsize=8)
plt.tight_layout()
path = FIGURES_DIR / 'per_class_f1_test.png'
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

---
## Cell 17 — Final Summary and Output File Listing


In [ ]:
import os

print('=' * 70)
print('TRAINING COMPLETE — FINAL SUMMARY')
print('=' * 70)
print()
print('── Training ──')
print(f'  Model             : {PRETRAINED}  (fine-tuned from pretrained weights)')
print(f'  Dataset           : QCRI/HumAID-all')
print(f'  Train size        : {len(splits["train"]):,}')
print(f'  Val size          : {len(splits["validation"]):,}  (used for early stopping only)')
print(f'  Test size         : {len(splits["test"]):,}  (evaluated once, after training)')
print(f'  Epochs run        : {history[-1]["epoch"]}  (max {EPOCHS}, patience {PATIENCE})')
print(f'  Best epoch        : {best_epoch}')
print(f'  Total train time  : {TOTAL_TRAIN_TIME:.1f} min')
print()
print('── Validation metrics (best epoch — NOT the test result) ──')
bvr = next(r for r in history if r['epoch'] == best_epoch)
print(f'  Val Macro F1      : {bvr["val_macro_f1"]:.4f}')
print(f'  Val Weighted F1   : {bvr["val_weighted_f1"]:.4f}')
print(f'  Val Accuracy      : {bvr["val_accuracy"]:.4f}')
print()
print('── TEST SET RESULTS (held-out, evaluated once) ──')
print(f'  Accuracy          : {test_metrics["accuracy"]:.4f}')
print(f'  Macro F1          : {test_metrics["macro_f1"]:.4f}  ← primary metric')
print(f'  Weighted F1       : {test_metrics["weighted_f1"]:.4f}')
print(f'  Macro Precision   : {test_metrics["macro_precision"]:.4f}')
print(f'  Macro Recall      : {test_metrics["macro_recall"]:.4f}')
print()
print('── Per-class F1 (test) ──')
for lbl, f1 in sorted(PER_CLASS_F1.items(), key=lambda x: -x[1]):
    print(f'  {lbl:<50} {f1:.4f}')
print()
print('── Output files ──')

# Walk the output directory and list every generated file
for root, dirs, files in os.walk(OUTPUT_DIR):
    dirs.sort()
    for fname in sorted(files):
        full   = Path(root) / fname
        rel    = full.relative_to(OUTPUT_DIR)
        size_kb = full.stat().st_size / 1024
        size_str = (f'{full.stat().st_size/1e6:.1f} MB'
                    if full.stat().st_size > 500_000
                    else f'{size_kb:.1f} KB')
        print(f'  {str(rel):<60}  {size_str}')

print()
print('All files saved under:', OUTPUT_DIR)
print('Download via Kaggle Output tab or use add_output() to expose as a dataset.')